In [67]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sklearn
import lightgbm as lgb
import time


from data_loader import split_data
from data_loader import test_data_func

from sklearn.model_selection import KFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error



In [68]:
path = r'data/train.csv'
setting = 'xy'

lst_1 = ['RoofMatl', 'HouseStyle', 'LandContour', 'LandSlope', 'Alley', 'BldgType', 'Street', 'MSZoning', 'RoofStyle', 'LowQualFinSF', 'Heating', 'BsmtFinType2', 'Utilities', 'Condition2', 'HeatingQC', 'Electrical', 'PoolArea']

X_train, X_val, y_train, y_val = split_data(path, setting, lst_1)


In [ ]:
tscv = KFold(n_splits=5)

params = {
    'objective': ['regression'],
    'metric': ['rmse'],
    'boosting_type': ['gbdt'],
    # --- Force Simpler Trees ---
    'num_leaves': [8, 12, 15, 20, 25],  # Keep leaves small (8 to 25)
    'max_depth': [3, 4, 5, 6],  # Shallow trees prevent memorization
    'min_child_samples': [20, 30, 40, 50],  # Require more samples per leaf
    # --- Add Regularization (CRITICAL for small data) ---
    'subsample': [0.6, 0.7, 0.8],  # Train each tree on 60-80% of rows
    'colsample_bytree': [0.4, 0.5, 0.6, 0.7],  # Use 40-70% of features per split
    'reg_alpha': [0.0, 0.1, 1.0, 5.0],  # L1 regularization
    'reg_lambda': [0.0, 0.1, 1.0, 5.0, 10.0],  # L2 regularization
    'learning_rate': [0.01, 0.03, 0.05],
}

estimator = lgb.LGBMRegressor()

random_search = RandomizedSearchCV(estimator=estimator, param_distributions=params, cv=tscv, n_iter=10, random_state=42, scoring='neg_mean_squared_error')

In [ ]:
search_model = random_search.fit(X_train, y_train,eval_set=[(X_val, y_val)], eval_metric='rmse', callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)])

In [77]:
y_pred = search_model.predict(X_val)
print(f'Best parameters: {search_model.best_params_}')
print(f'MSE: {mean_squared_error(y_val, y_pred)}')

Best parameters: {'objective': 'regression', 'num_leaves': 46, 'n_estimators': 211, 'min_child_samples': 27, 'metric': 'rmse', 'max_depth': 6, 'learning_rate': np.float64(0.044984326689694466), 'boosting_type': 'gbdt'}
MSE: 0.017893770225393735


In [79]:
y_true, cat_cols = test_data_func(r'data/test.csv', lst_1)

ids = y_true['Id'].to_list()

predictions = search_model.predict(y_true.drop(['Id'], axis=1))
predictions = np.round(np.expm1(predictions), -2)
predictions

result = pd.DataFrame({'Id':ids, 'SalePrice':predictions})

nr = 7

result.to_csv(fr'results/results_{nr}.csv', index=False)

### Plotting

In [ ]:
# 3. Extract search results into DataFrame
results = pd.DataFrame(search_model.cv_results_)

# Flip negative RMSE back to positive
results['rmse'] = -results['mean_test_score']

# 4. Plot RMSE progression over iterations
plt.figure(figsize=(10, 5))
plt.plot(
    results.index,
    results['rmse'],
    marker='o',
    linestyle='--',
    color='#1f77b4',
    label='Trial RMSE',
)

# Highlight the best trial
best_iter = results['rmse'].idxmin()
best_rmse = results.loc[best_iter, 'rmse']
plt.scatter(
    [best_iter],
    [best_rmse],
    color='red',
    s=120,
    zorder=5,
    label=f'Best Trial (RMSE: {best_rmse:.4f})',
)

plt.title(
    'Random Search Tuning Progression (Validation RMSE)',
    fontsize=13,
    weight='bold',
)
plt.xlabel('Random Search Iteration (Trial Number)', fontsize=11)
plt.ylabel('Validation RMSE (Lower is Better)', fontsize=11)
plt.xticks(results.index)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()